# Test of the source-hour manifest (`07_source_hour_manifest`)

This notebook tests `07_source_hour_manifest` with a small hand-made Bronze table in the schema `github_lakehouse.test_gold_manifest`. It runs the real notebook with `%run` and `test_schema = test_gold_manifest`, so **production tables are never written**. The last check section proves that.

**Before you run it:** run `08_gold_audit_table` once. The test copies the definition of the audit table from it.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** Their headings say so. After each of them, continue with "Run all below" from the next cell.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: an empty Bronze table gives `NO_INPUT` |
| Schema | T7: the manifest has exactly the columns and types of the contract |
| Exact values | T7: hand-made rows give the hand-calculated manifest |
| Missing values | T7: an hour without a file. T11: an hour without S3 evidence |
| Rerun | T8, T9: a second run changes nothing |
| S3 evidence | T10, T11: evidence from an earlier comparison is kept and compared again |
| Late data | T12, T13: more rows for an old hour, a new hour and a new day |
| Removed data | T14, T15: a removed file and a removed day |
| Failed check | T16 to T18: a failing quality gate leaves the manifest unchanged, then a recovery run |
| Bad parameter | T19 to T21: an invalid `source_date`, `test_schema` or `check_s3` stops the notebook before any write |
| Production safety | T22: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean. The Bronze test table has the five columns of the real Bronze table. The audit test table is an empty copy of the real audit table.

The manifest test table is not created here. The notebook under test creates it, which also tests its `CREATE TABLE` statement.

In [0]:
TEST_SCHEMA = "test_gold_manifest"

TEST_BRONZE = f"github_lakehouse.{TEST_SCHEMA}.github_events_raw"
TEST_MANIFEST = f"github_lakehouse.{TEST_SCHEMA}.source_hour_manifest"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.source_hour_manifest_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.manifest_after_run_1"

PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_MANIFEST = "github_lakehouse.audit.source_hour_manifest"
PROD_STAGING = "github_lakehouse.gold_staging.source_hour_manifest"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json           STRING,
        _source_file       STRING,
        _source_file_name  STRING,
        _source_date       DATE,
        _ingested_at       TIMESTAMP
    )
    USING DELTA
""")

spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The production manifest and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_MANIFEST):
    prod_manifest_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_MANIFEST} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_manifest_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version   :", prod_audit_version_before)
print("Production manifest version:", prod_manifest_version_before)
print("Production staging version :", prod_staging_version_before)

## T3. Run on the empty Bronze table

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-dry-run" $check_s3=""

## T4. Checks: no input

An empty Bronze table has no source dates, so no hour is expected. The run must end as `NO_INPUT`, publish nothing and still write its audit row.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test Bronze table' AS check_name,
               '{BRONZE_TABLE}' AS actual, '{TEST_BRONZE}' AS expected
        UNION ALL
        SELECT '02 the notebook wrote the test manifest', '{MANIFEST_TABLE}', '{TEST_MANIFEST}'
        UNION ALL
        SELECT '03 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '04 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '05 manifest rows', CAST(COUNT(*) AS STRING), '0' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '06 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '07 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '08 audit pipeline_name', MAX(pipeline_name), 'gold_source_hour_manifest'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '09 audit gold_table', MAX(gold_table), '{TEST_MANIFEST}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '10 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '11 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '12 audit expected_hours', CAST(MAX(quality_metrics['expected_hours']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '13 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made Bronze rows

Two source dates, so 48 hours are expected. Only eight of them have a file.

| File | Source date | Rows | What it tests |
|---|---|---|---|
| `2024-02-01-0.json.gz` | 1 Feb | 3 | First hour. Two ingestion times: the earlier one must be kept |
| `2024-02-01-1.json.gz` | 1 Feb | 2 | One row is not JSON at all |
| `2024-02-01-10.json.gz` | 1 Feb | 1 | Hour with two digits. The row has no `type` |
| `2024-02-01-23.json.gz` | 1 Feb | 1 | Last hour of a day |
| `2024-02-02-0.json.gz` | 2 Feb | 1 | First hour of the next day |
| `2024-02-02-21.json.gz` | 2 Feb | 4 | Neighbour of the thin hour |
| `2024-02-02-22.json.gz` | 2 Feb | 1 | Thin hour: 1 row between two hours with 4 rows, which is 25% |
| `2024-02-02-23.json.gz` | 2 Feb | 4 | Last expected hour |
| `notes.json.gz` | 2 Feb | 1 | A file that no hour expects |

18 rows in total. Two of them have no readable event type.

In [0]:
spark.sql("INSERT INTO " + TEST_BRONZE + """ VALUES
    ('{"id":"a1","type":"PushEvent","created_at":"2024-02-01T00:05:10Z"}',    's3://test/2024-02-01-0.json.gz',  '2024-02-01-0.json.gz',  DATE'2024-02-01', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"a2","type":"WatchEvent","created_at":"2024-02-01T00:05:40Z"}',   's3://test/2024-02-01-0.json.gz',  '2024-02-01-0.json.gz',  DATE'2024-02-01', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"a3","type":"CreateEvent","created_at":"2024-02-01T00:59:59Z"}',  's3://test/2024-02-01-0.json.gz',  '2024-02-01-0.json.gz',  DATE'2024-02-01', TIMESTAMP'2024-03-02 09:00:00'),
    ('{"id":"b1","type":"IssuesEvent","created_at":"2024-02-01T01:10:00Z"}',  's3://test/2024-02-01-1.json.gz',  '2024-02-01-1.json.gz',  DATE'2024-02-01', TIMESTAMP'2024-03-01 08:00:00'),
    ('this is not json',                                                      's3://test/2024-02-01-1.json.gz',  '2024-02-01-1.json.gz',  DATE'2024-02-01', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"c1","created_at":"2024-02-01T10:30:00Z"}',                       's3://test/2024-02-01-10.json.gz', '2024-02-01-10.json.gz', DATE'2024-02-01', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"d1","type":"ForkEvent","created_at":"2024-02-01T23:59:59Z"}',    's3://test/2024-02-01-23.json.gz', '2024-02-01-23.json.gz', DATE'2024-02-01', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"e1","type":"ReleaseEvent","created_at":"2024-02-02T00:00:00Z"}', 's3://test/2024-02-02-0.json.gz',  '2024-02-02-0.json.gz',  DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"g1","type":"PushEvent","created_at":"2024-02-02T21:00:05Z"}',    's3://test/2024-02-02-21.json.gz', '2024-02-02-21.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"g2","type":"PushEvent","created_at":"2024-02-02T21:00:45Z"}',    's3://test/2024-02-02-21.json.gz', '2024-02-02-21.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"g3","type":"PushEvent","created_at":"2024-02-02T21:30:00Z"}',    's3://test/2024-02-02-21.json.gz', '2024-02-02-21.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"g4","type":"PushEvent","created_at":"2024-02-02T21:59:00Z"}',    's3://test/2024-02-02-21.json.gz', '2024-02-02-21.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"h1","type":"PushEvent","created_at":"2024-02-02T22:15:30Z"}',    's3://test/2024-02-02-22.json.gz', '2024-02-02-22.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"f1","type":"PushEvent","created_at":"2024-02-02T23:01:00Z"}',    's3://test/2024-02-02-23.json.gz', '2024-02-02-23.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"f2","type":"PushEvent","created_at":"2024-02-02T23:02:00Z"}',    's3://test/2024-02-02-23.json.gz', '2024-02-02-23.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"f3","type":"PushEvent","created_at":"2024-02-02T23:03:00Z"}',    's3://test/2024-02-02-23.json.gz', '2024-02-02-23.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"f4","type":"PushEvent","created_at":"2024-02-02T23:04:00Z"}',    's3://test/2024-02-02-23.json.gz', '2024-02-02-23.json.gz', DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00'),
    ('{"id":"x1","type":"PushEvent","created_at":"2024-02-02T12:00:00Z"}',    's3://test/notes.json.gz',         'notes.json.gz',         DATE'2024-02-02', TIMESTAMP'2024-03-01 08:00:00')
""")

print("Bronze test rows:", spark.table(TEST_BRONZE).count())

### A one-line summary of every manifest row

The checks below compare whole manifest rows. This view writes the important columns of a row into one readable line, for example:

`rows=3 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true`

The view is not a copy. It always reads the manifest as it is at that moment.

In [0]:
manifest_summary = spark.sql(f"""
    SELECT
        expected_file_name,
        concat(
            'rows=',      CAST(bronze_row_count AS STRING),
            ' ingested=', CAST(bronze_ingested AS STRING),
            ' s3_found=', COALESCE(CAST(s3_file_found AS STRING), 'NULL'),
            ' s3_lines=', COALESCE(CAST(s3_line_count AS STRING), 'NULL'),
            ' match=',    COALESCE(CAST(counts_match AS STRING), 'NULL'),
            ' evidence=', evidence_level,
            ' complete=', CAST(is_complete AS STRING)
        ) AS state
    FROM {TEST_MANIFEST}
""")
manifest_summary.createOrReplaceTempView("manifest_summary")

## T6. Run 1: all dates

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-run1" $check_s3=""

## T7. Checks of run 1: schema and exact values

Worked out by hand:

- 2 days, so 48 expected hours. 8 hours have a file, so 8 are complete and 40 are not.
- Nothing was compared with S3, so every row has the evidence level `BRONZE_ONLY` and no S3 values.
- `2024-02-01-0.json.gz` has 3 rows. Its first ingestion time is the earlier of its two times.
- `2024-02-01-5.json.gz` has no file: 0 rows, not ingested, not complete.
- `notes.json.gz` is in Bronze but no hour expects it.
- `2024-02-02-22.json.gz` has 1 row between two hours with 4 rows: 25%, a thin hour.
- 18 Bronze rows were checked for a readable type. 2 have none.

In [0]:
actual_schema = spark.table(TEST_MANIFEST).schema.simpleString()

expected_schema = (
    "struct<source_hour_utc:timestamp,source_date:date,expected_file_name:string,"
    "bronze_ingested:boolean,bronze_row_count:bigint,bronze_first_ingested_at:timestamp,"
    "s3_file_found:boolean,s3_line_count:bigint,counts_match:boolean,evidence_level:string,"
    "is_complete:boolean,checked_at:timestamp,run_id:string>"
)

print("Actual  :", actual_schema)
print("Expected:", expected_schema)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the manifest' AS check_name,
               '{actual_schema}' AS actual, '{expected_schema}' AS expected
        UNION ALL
        SELECT '02 manifest rows', CAST(COUNT(*) AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '03 different hours', CAST(COUNT(DISTINCT source_hour_utc) AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '04 fewest and most hours of a day', concat(CAST(MIN(hours) AS STRING), ' and ', CAST(MAX(hours) AS STRING)), '24 and 24'
        FROM (SELECT source_date, COUNT(*) AS hours FROM {TEST_MANIFEST} GROUP BY source_date)
        UNION ALL
        SELECT '05 first hour', CAST(MIN(source_hour_utc) AS STRING), '2024-02-01 00:00:00' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '06 last hour', CAST(MAX(source_hour_utc) AS STRING), '2024-02-02 23:00:00' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '07 complete hours', CAST(count_if(is_complete) AS STRING), '8' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '08 hours that are not complete', CAST(count_if(NOT is_complete) AS STRING), '40' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '09 rows with evidence BRONZE_ONLY', CAST(count_if(evidence_level = 'BRONZE_ONLY') AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '10 rows with any S3 value', CAST(count_if(s3_file_found IS NOT NULL OR s3_line_count IS NOT NULL OR counts_match IS NOT NULL) AS STRING), '0' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '11 rows written by this run', CAST(count_if(run_id = 'test-run1' AND checked_at IS NOT NULL) AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '12 state of 2024-02-01-0', MAX(state), 'rows=3 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '13 hour of 2024-02-01-0', CAST(MAX(source_hour_utc) AS STRING), '2024-02-01 00:00:00'
        FROM {TEST_MANIFEST} WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '14 date of 2024-02-01-0', CAST(MAX(source_date) AS STRING), '2024-02-01'
        FROM {TEST_MANIFEST} WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '15 first ingestion of 2024-02-01-0', CAST(MAX(bronze_first_ingested_at) AS STRING), '2024-03-01 08:00:00'
        FROM {TEST_MANIFEST} WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '16 state of 2024-02-01-1', MAX(state), 'rows=2 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-1.json.gz'
        UNION ALL
        SELECT '17 hour of 2024-02-01-10', CAST(MAX(source_hour_utc) AS STRING), '2024-02-01 10:00:00'
        FROM {TEST_MANIFEST} WHERE expected_file_name = '2024-02-01-10.json.gz'
        UNION ALL
        SELECT '18 state of 2024-02-01-10', MAX(state), 'rows=1 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-10.json.gz'
        UNION ALL
        SELECT '19 state of 2024-02-01-5 (no file)', MAX(state), 'rows=0 ingested=false s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=false'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-5.json.gz'
        UNION ALL
        SELECT '20 first ingestion of 2024-02-01-5 (no file)', CAST(MAX(bronze_first_ingested_at) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_MANIFEST} WHERE expected_file_name = '2024-02-01-5.json.gz'
        UNION ALL
        SELECT '21 state of 2024-02-02-23', MAX(state), 'rows=4 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-02-23.json.gz'
        UNION ALL
        SELECT '22 notes.json.gz is not in the manifest', CAST(COUNT(*) AS STRING), '0'
        FROM {TEST_MANIFEST} WHERE expected_file_name = 'notes.json.gz'
        UNION ALL
        SELECT '23 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '24 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '25 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '26 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '27 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '28 audit Bronze rows read', CAST(MAX(source_rows_read['bronze']) AS STRING), '18'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '29 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '48'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '30 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '31 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '32 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '33 audit expected_hours', CAST(MAX(quality_metrics['expected_hours']) AS STRING), '48'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '34 audit complete_hours', CAST(MAX(quality_metrics['complete_hours']) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '35 audit incomplete_hours', CAST(MAX(quality_metrics['incomplete_hours']) AS STRING), '40'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '36 audit hours_in_bronze', CAST(MAX(quality_metrics['hours_in_bronze']) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '37 audit hours_compared_with_s3', CAST(MAX(quality_metrics['hours_compared_with_s3']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '38 audit hours_with_count_mismatch', CAST(MAX(quality_metrics['hours_with_count_mismatch']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '39 audit unexpected_bronze_files', CAST(MAX(quality_metrics['unexpected_bronze_files']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '40 audit thin_hours', CAST(MAX(quality_metrics['thin_hours']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '41 audit bronze_rows_checked_for_type', CAST(MAX(quality_metrics['bronze_rows_checked_for_type']) AS STRING), '18'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '42 audit bronze_rows_without_readable_type', CAST(MAX(quality_metrics['bronze_rows_without_readable_type']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '43 audit s3_check_requested', CAST(MAX(quality_metrics['s3_check_requested']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '44 audit s3_check_done', CAST(MAX(quality_metrics['s3_check_done']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again

First the manifest of run 1 is saved as a copy, so that run 2 can be compared with it.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT * FROM {TEST_MANIFEST}
""")

print("Copy of the manifest after run 1:", TEST_COPY)

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-run2" $check_s3=""

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns `checked_at` and `run_id` are left out, because they are new in every run.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 manifest rows' AS check_name, CAST(COUNT(*) AS STRING) AS actual, '48' AS expected
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '02 rows of run 2 that differ from run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT source_hour_utc, source_date, expected_file_name, bronze_ingested, bronze_row_count,
                   bronze_first_ingested_at, s3_file_found, s3_line_count, counts_match, evidence_level, is_complete
            FROM {TEST_MANIFEST}
            EXCEPT
            SELECT source_hour_utc, source_date, expected_file_name, bronze_ingested, bronze_row_count,
                   bronze_first_ingested_at, s3_file_found, s3_line_count, counts_match, evidence_level, is_complete
            FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '03 rows of run 1 that are missing after run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT source_hour_utc, source_date, expected_file_name, bronze_ingested, bronze_row_count,
                   bronze_first_ingested_at, s3_file_found, s3_line_count, counts_match, evidence_level, is_complete
            FROM {TEST_COPY}
            EXCEPT
            SELECT source_hour_utc, source_date, expected_file_name, bronze_ingested, bronze_row_count,
                   bronze_first_ingested_at, s3_file_found, s3_line_count, counts_match, evidence_level, is_complete
            FROM {TEST_MANIFEST}
        )
        UNION ALL
        SELECT '04 rows written by run 2', CAST(count_if(run_id = 'test-run2') AS STRING), '48'
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '05 audit status of run 2', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '06 audit rows so far', CAST(COUNT(*) AS STRING), '3'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: S3 evidence from an earlier comparison, and one source date

A test cannot read the real raw files. So the test writes S3 evidence into the manifest by hand, as if an earlier run with `check_s3 = yes` had found it. The next run must keep that evidence and compare it with Bronze again.

| Hour | S3 evidence written by hand | Bronze rows | Expected result |
|---|---|---|---|
| `2024-02-01-0` | found, 3 lines | 3 | Counts match. Complete |
| `2024-02-01-1` | found, 5 lines | 2 | Counts differ. Not complete |
| `2024-02-01-10` | not found | 1 | Nothing to compare with. Complete on the Bronze evidence |
| `2024-02-01-5` | found, 7 lines | 0 | In S3 but not in Bronze. Not complete |

The run also gets `source_date = 2024-02-02`. The manifest is still rebuilt for both days, but the routing check reads only the 11 Bronze rows of 2 February.

In [0]:
spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET s3_file_found = true, s3_line_count = 3
    WHERE expected_file_name = '2024-02-01-0.json.gz'
""")

spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET s3_file_found = true, s3_line_count = 5
    WHERE expected_file_name = '2024-02-01-1.json.gz'
""")

spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET s3_file_found = false
    WHERE expected_file_name = '2024-02-01-10.json.gz'
""")

spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET s3_file_found = true, s3_line_count = 7
    WHERE expected_file_name = '2024-02-01-5.json.gz'
""")

print("S3 evidence written by hand for four hours.")

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="2024-02-02" $run_id="test-run3" $check_s3=""

## T11. Checks of run 3

Worked out by hand: of the 8 hours with a file, `2024-02-01-1` is no longer complete, so 7 hours are complete and 41 are not. Three hours have a raw file that was found. Two of them have different counts.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 manifest rows' AS check_name, CAST(COUNT(*) AS STRING) AS actual, '48' AS expected
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '02 state of 2024-02-01-0 (counts match)', MAX(state), 'rows=3 ingested=true s3_found=true s3_lines=3 match=true evidence=BRONZE_AND_S3 complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '03 state of 2024-02-01-1 (counts differ)', MAX(state), 'rows=2 ingested=true s3_found=true s3_lines=5 match=false evidence=BRONZE_AND_S3 complete=false'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-1.json.gz'
        UNION ALL
        SELECT '04 state of 2024-02-01-10 (not found in S3)', MAX(state), 'rows=1 ingested=true s3_found=false s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-10.json.gz'
        UNION ALL
        SELECT '05 state of 2024-02-01-5 (in S3, not in Bronze)', MAX(state), 'rows=0 ingested=false s3_found=true s3_lines=7 match=false evidence=BRONZE_AND_S3 complete=false'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-5.json.gz'
        UNION ALL
        SELECT '06 state of 2024-02-01-23 (never compared)', MAX(state), 'rows=1 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-23.json.gz'
        UNION ALL
        SELECT '07 complete hours', CAST(count_if(is_complete) AS STRING), '7' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '08 hours that are not complete', CAST(count_if(NOT is_complete) AS STRING), '41' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '09 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '10 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '11 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-02'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '12 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '13 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '48'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '14 audit complete_hours', CAST(MAX(quality_metrics['complete_hours']) AS STRING), '7'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '15 audit hours_compared_with_s3', CAST(MAX(quality_metrics['hours_compared_with_s3']) AS STRING), '3'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '16 audit hours_with_count_mismatch', CAST(MAX(quality_metrics['hours_with_count_mismatch']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '17 audit bronze_rows_checked_for_type (2 February only)', CAST(MAX(quality_metrics['bronze_rows_checked_for_type']) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '18 audit bronze_rows_without_readable_type', CAST(MAX(quality_metrics['bronze_rows_without_readable_type']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '19 audit Bronze rows read', CAST(MAX(source_rows_read['bronze']) AS STRING), '18'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: late data

Four new Bronze rows arrive:

| File | Source date | New rows | What it tests |
|---|---|---|---|
| `2024-02-01-1.json.gz` | 1 Feb | 1 | An old hour grows from 2 to 3 rows. S3 still says 5, so it stays not complete |
| `2024-02-01-2.json.gz` | 1 Feb | 2 | An hour that was missing arrives |
| `2024-02-03-0.json.gz` | 3 Feb | 1 | A new day: 24 more hours are expected |

In [0]:
spark.sql("INSERT INTO " + TEST_BRONZE + """ VALUES
    ('{"id":"b3","type":"PushEvent","created_at":"2024-02-01T01:20:00Z"}',  's3://test/2024-02-01-1.json.gz', '2024-02-01-1.json.gz', DATE'2024-02-01', TIMESTAMP'2024-03-05 08:00:00'),
    ('{"id":"i1","type":"PushEvent","created_at":"2024-02-01T02:01:00Z"}',  's3://test/2024-02-01-2.json.gz', '2024-02-01-2.json.gz', DATE'2024-02-01', TIMESTAMP'2024-03-05 08:00:00'),
    ('{"id":"i2","type":"PushEvent","created_at":"2024-02-01T02:02:00Z"}',  's3://test/2024-02-01-2.json.gz', '2024-02-01-2.json.gz', DATE'2024-02-01', TIMESTAMP'2024-03-05 08:00:00'),
    ('{"id":"j1","type":"WatchEvent","created_at":"2024-02-03T00:10:00Z"}', 's3://test/2024-02-03-0.json.gz', '2024-02-03-0.json.gz', DATE'2024-02-03', TIMESTAMP'2024-03-05 08:00:00')
""")

print("Bronze test rows:", spark.table(TEST_BRONZE).count())

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-run4" $check_s3=""

## T13. Checks of run 4

Worked out by hand: 3 days, so 72 expected hours. 10 hours have a file. `2024-02-01-1` is still not complete, so 9 hours are complete and 63 are not. Bronze now has 22 rows.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 manifest rows' AS check_name, CAST(COUNT(*) AS STRING) AS actual, '72' AS expected
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '02 different hours', CAST(COUNT(DISTINCT source_hour_utc) AS STRING), '72' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '03 fewest and most hours of a day', concat(CAST(MIN(hours) AS STRING), ' and ', CAST(MAX(hours) AS STRING)), '24 and 24'
        FROM (SELECT source_date, COUNT(*) AS hours FROM {TEST_MANIFEST} GROUP BY source_date)
        UNION ALL
        SELECT '04 last hour', CAST(MAX(source_hour_utc) AS STRING), '2024-02-03 23:00:00' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '05 state of 2024-02-01-1 (3 rows now, S3 says 5)', MAX(state), 'rows=3 ingested=true s3_found=true s3_lines=5 match=false evidence=BRONZE_AND_S3 complete=false'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-1.json.gz'
        UNION ALL
        SELECT '06 state of 2024-02-01-2 (arrived late)', MAX(state), 'rows=2 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-2.json.gz'
        UNION ALL
        SELECT '07 state of 2024-02-03-0 (new day)', MAX(state), 'rows=1 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-03-0.json.gz'
        UNION ALL
        SELECT '08 state of 2024-02-03-1 (new day, no file)', MAX(state), 'rows=0 ingested=false s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=false'
        FROM manifest_summary WHERE expected_file_name = '2024-02-03-1.json.gz'
        UNION ALL
        SELECT '09 state of 2024-02-01-0 (unchanged)', MAX(state), 'rows=3 ingested=true s3_found=true s3_lines=3 match=true evidence=BRONZE_AND_S3 complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '10 complete hours', CAST(count_if(is_complete) AS STRING), '9' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '11 hours that are not complete', CAST(count_if(NOT is_complete) AS STRING), '63' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '12 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '13 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-03]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '14 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '72'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '15 audit Bronze rows read', CAST(MAX(source_rows_read['bronze']) AS STRING), '22'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '16 audit hours_in_bronze', CAST(MAX(quality_metrics['hours_in_bronze']) AS STRING), '10'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '17 audit thin_hours', CAST(MAX(quality_metrics['thin_hours']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '18 audit bronze_rows_checked_for_type', CAST(MAX(quality_metrics['bronze_rows_checked_for_type']) AS STRING), '22'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '19 audit bronze_rows_without_readable_type', CAST(MAX(quality_metrics['bronze_rows_without_readable_type']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: removed data

Two things are removed from Bronze:

- all rows of `2024-02-01-0.json.gz`. The hour stays expected, but it has no file any more.
- all rows of 3 February. The last source date is 2 February again, so the 24 hours of 3 February must disappear from the manifest.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_BRONZE}
    WHERE _source_file_name = '2024-02-01-0.json.gz'
""")

spark.sql(f"""
    DELETE FROM {TEST_BRONZE}
    WHERE _source_date = DATE'2024-02-03'
""")

print("Bronze test rows:", spark.table(TEST_BRONZE).count())

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-run5" $check_s3=""

## T15. Checks of run 5

Worked out by hand: 48 expected hours again. 8 hours have a file. `2024-02-01-1` still has different counts, so 7 hours are complete and 41 are not. The S3 evidence of `2024-02-01-0` is kept: S3 says 3 lines, Bronze has 0 rows now, so the counts differ. Bronze has 18 rows.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 manifest rows' AS check_name, CAST(COUNT(*) AS STRING) AS actual, '48' AS expected
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '02 rows of 3 February', CAST(count_if(source_date = DATE'2024-02-03') AS STRING), '0' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '03 last hour', CAST(MAX(source_hour_utc) AS STRING), '2024-02-02 23:00:00' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '04 state of 2024-02-01-0 (file removed, S3 evidence kept)', MAX(state), 'rows=0 ingested=false s3_found=true s3_lines=3 match=false evidence=BRONZE_AND_S3 complete=false'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '05 first ingestion of 2024-02-01-0 (file removed)', CAST(MAX(bronze_first_ingested_at) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_MANIFEST} WHERE expected_file_name = '2024-02-01-0.json.gz'
        UNION ALL
        SELECT '06 state of 2024-02-01-2 (unchanged)', MAX(state), 'rows=2 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-2.json.gz'
        UNION ALL
        SELECT '07 complete hours', CAST(count_if(is_complete) AS STRING), '7' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '08 hours that are not complete', CAST(count_if(NOT is_complete) AS STRING), '41' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '09 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '11 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '48'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '12 audit Bronze rows read', CAST(MAX(source_rows_read['bronze']) AS STRING), '18'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '13 audit hours_in_bronze', CAST(MAX(quality_metrics['hours_in_bronze']) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '14 audit hours_compared_with_s3', CAST(MAX(quality_metrics['hours_compared_with_s3']) AS STRING), '3'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '15 audit hours_with_count_mismatch', CAST(MAX(quality_metrics['hours_with_count_mismatch']) AS STRING), '3'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: a quality gate fails

The test puts a second copy of the row `2024-02-01-1.json.gz` into the manifest. That hour has S3 evidence, so the next run finds the evidence twice and builds two rows for one hour. Three gates must catch it: the row count, the unique hour and the unique file name.

The cell below prepares the fault. The run itself is in the cell after it.

In [0]:
spark.sql(f"""
    INSERT INTO {TEST_MANIFEST}
    SELECT * FROM {TEST_MANIFEST}
    WHERE expected_file_name = '2024-02-01-1.json.gz'
""")

print("Manifest rows with the extra copy:", spark.table(TEST_MANIFEST).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: Source-hour manifest FAILED: Quality gates failed: ...`. After it has failed, continue with "Run all below" from the cell after it.

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-run6-fail" $check_s3=""

## T17. Checks of run 6: the manifest is unchanged

The run must have written its audit row with the status `FAILED` and the names of the three gates. The manifest must still be exactly what it was before the run: 49 rows, all written by run 5. The rejected result is only in the staging table.

After the checks, the test removes both copies of the row, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'ROW_COUNT_IS_NOT_24_PER_DAY; KEY_NOT_UNIQUE; FILE_NAME_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: ROW_COUNT_IS_NOT_24_PER_DAY; KEY_NOT_UNIQUE; FILE_NAME_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 manifest rows (unchanged)', CAST(COUNT(*) AS STRING), '49'
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '07 manifest rows written by the failed run', CAST(count_if(run_id = 'test-run6-fail') AS STRING), '0'
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '08 manifest rows still from run 5', CAST(count_if(run_id = 'test-run5') AS STRING), '49'
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '09 staging rows of the rejected result', CAST(count_if(run_id = 'test-run6-fail') AS STRING), '49'
        FROM {TEST_STAGING}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_MANIFEST}
    WHERE expected_file_name = '2024-02-01-1.json.gz'
""")

print("Manifest rows after the repair:", spark.table(TEST_MANIFEST).count())

## T18. Run 7: recovery

The same run again, now without the fault. The S3 evidence of `2024-02-01-1` was deleted together with the two copies, so this hour is judged on its Bronze evidence alone and is complete again.

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-run7" $check_s3=""

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 manifest rows', CAST(COUNT(*) AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '03 different hours', CAST(COUNT(DISTINCT source_hour_utc) AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '04 rows written by run 7', CAST(count_if(run_id = 'test-run7') AS STRING), '48' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '05 state of 2024-02-01-1 (no S3 evidence any more)', MAX(state), 'rows=3 ingested=true s3_found=NULL s3_lines=NULL match=NULL evidence=BRONZE_ONLY complete=true'
        FROM manifest_summary WHERE expected_file_name = '2024-02-01-1.json.gz'
        UNION ALL
        SELECT '06 complete hours', CAST(count_if(is_complete) AS STRING), '8' FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '07 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '08 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '09 audit hours_compared_with_s3', CAST(MAX(quality_metrics['hours_compared_with_s3']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '10 audit rows so far', CAST(COUNT(*) AS STRING), '8' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, continue with "Run all below" from the cell after it.

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="2024-2-3" $run_id="test-bad-date" $check_s3=""

## T20. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, continue with "Run all below" from the cell after it.

In [0]:
%run ../07_source_hour_manifest $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema" $check_s3=""

## T21. Bad parameter: `check_s3` in a test

`check_s3 = yes` reads the real raw files, so it is not allowed together with a test schema.

**The next cell is EXPECTED to fail** with `ValueError: check_s3 = yes reads the real raw files in S3`. After it has failed, continue with "Run all below" from the cell after it.

In [0]:
%run ../07_source_hour_manifest $test_schema="test_gold_manifest" $source_date="" $run_id="test-bad-s3" $check_s3="yes"

## T22. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still has 8 rows and the manifest is still the one of run 7.
- The versions of the production tables are read again and compared with the versions from T2.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_MANIFEST):
    prod_manifest_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_MANIFEST} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_manifest_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version   :", prod_audit_version_after)
print("Production manifest version:", prod_manifest_version_after)
print("Production staging version :", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit rows (no row from the three stopped runs)' AS check_name, CAST(COUNT(*) AS STRING) AS actual, '8' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'test-bad-s3')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 manifest rows still from run 7', CAST(count_if(run_id = 'test-run7') AS STRING), '48'
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '04 manifest rows', CAST(COUNT(*) AS STRING), '48'
        FROM {TEST_MANIFEST}
        UNION ALL
        SELECT '05 audit rows by status',
               concat('SUCCEEDED=', CAST(count_if(status = 'SUCCEEDED') AS STRING),
                      ' FAILED=', CAST(count_if(status = 'FAILED') AS STRING),
                      ' NO_INPUT=', CAST(count_if(status = 'NO_INPUT') AS STRING)),
               'SUCCEEDED=6 FAILED=1 NO_INPUT=1'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production manifest version unchanged', '{prod_manifest_version_after}', '{prod_manifest_version_before}'
        UNION ALL
        SELECT '08 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
if failed_count > 0:
    raise AssertionError(f"T22: {failed_count} check(s) failed. See the table above.")
print("T22: ALL CHECKS PASSED")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)
print("TEST FINISHED: every section passed.")